# Thermodynamic characterisation

Primer3 is optional at import time and is only required when this function is
invoked. The exact parameter set is included in every result.

In [ ]:
#| default_exp thermo

In [ ]:
#| export
from __future__ import annotations

from dataclasses import dataclass

from probeharbour.models import Candidate, Characterization
from probeharbour.sequence import gc_percent, has_5g_repeat, has_simple_repeat

In [ ]:
#| export
@dataclass(frozen=True)
class ThermoParameters:
    parameters_id: str = "primer3-xenium-schildkraut-v1"
    mv_conc: float = 80.0
    dv_conc: float = 5.0
    dntp_conc: float = 0.0
    dna_conc: float = 1500.0
    salt_corrections_method: str = "schildkraut"
    structure_temp_c: float = 50.0

    def as_tm_primer3(self) -> dict[str, float | str]:
        return {"mv_conc": self.mv_conc, "dv_conc": self.dv_conc,
                "dntp_conc": self.dntp_conc, "dna_conc": self.dna_conc,
                "salt_corrections_method": self.salt_corrections_method}

    def as_structure_primer3(self) -> dict[str, float]:
        return {"mv_conc": self.mv_conc, "dv_conc": self.dv_conc,
                "dntp_conc": self.dntp_conc, "dna_conc": self.dna_conc,
                "temp_c": self.structure_temp_c}

    def as_primer3(self) -> dict[str, float | str]:
        """Backward-compatible alias for Tm parameters."""
        return self.as_tm_primer3()

In [ ]:
#| export
def characterize(candidate: Candidate, parameters: ThermoParameters = ThermoParameters()) -> Characterization:
    """Characterize a candidate using the optional Primer3 backend."""
    try:
        import primer3
    except ImportError as exc:
        raise RuntimeError("Primer3 support requires the optional primer3-py dependency") from exc
    tm_kwargs = parameters.as_tm_primer3()
    structure_kwargs = parameters.as_structure_primer3()
    rbd5, rbd3, binder = candidate.rbd5, candidate.rbd3, candidate.binder
    return Characterization(
        candidate_id=candidate.candidate_id,
        tm_5p=round(float(primer3.calc_tm(rbd5, **tm_kwargs)), 1),
        tm_3p=round(float(primer3.calc_tm(rbd3, **tm_kwargs)), 1),
        tm_full=round(float(primer3.calc_tm(binder, **tm_kwargs)), 1),
        gc_percent=gc_percent(binder),
        p3_hairpin_dg=round(float(primer3.calc_hairpin(binder, **structure_kwargs).dg) / 1000, 3),
        p3_homodimer_dg=round(float(primer3.calc_homodimer(binder, **structure_kwargs).dg) / 1000, 3),
        p3_heterodimer_dg=round(float(primer3.calc_heterodimer(rbd5, rbd3, **structure_kwargs).dg) / 1000, 3),
        has_simple_repeat=has_simple_repeat(binder),
        has_5g_repeat=has_5g_repeat(rbd5, rbd3),
        parameters_id=parameters.parameters_id,
    )